# C4 Traceability Intelligence: reranker and stage-1 analysis (reproducible)

**Component 4, Thuverakan T. (IT23281332).** This notebook re-runs, on this computer, the analysis that was done after the Colab
experiments, and prints the results in tables so they can be checked in the viva.

**What is reproduced exactly (deterministic, no GPU, no LLM, no internet needed once the models are cached):**
1. dataset sizes and the train / test split,
2. the stage-1 retrieval baseline (TF-IDF + MiniLM),
3. stage-1 recall with three embedding models (MiniLM, a code-search model, UniXcoder),
4. the *untrained* cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) fused with stage 1 + paired bootstrap,
5. the PetClinic comparison (stage 1 versus stage 1 + reranker), replaying the stored Llama verdicts,
6. the query-expansion experiment (replaying the stored LLM answers),
7. the C4 evaluation results already in the repository (retrieval, precision/recall, decay injection, NFRs).

**What is only *replayed* (recorded from Colab, cannot be recomputed here without the Colab files):** the fine-tuned CodeBERT and
fine-tuned MiniLM cross-encoders. They were trained on a T4 GPU; their result tables are stored in `data/results/colab_results.json`.
A local re-training cell is included (switched off by default); its numbers may differ slightly from Colab (different hardware and seeds).

Run it with the project's `.venv` kernel, from anywhere inside the project. Flags in the next cell switch the slower parts on or off.


**The outputs saved in this notebook were produced with `C4_FULL=1` (all 7 datasets) on an Apple-GPU Mac in about 3.5 minutes. With the default flags (`C4_FULL=0`) the run takes about 1.5 minutes and section 3 shows 4 datasets. The `(rec. ...)` numbers in section 3 are the values stored earlier in `data/results/stage1_embeddings.json`; they should equal the numbers of your run.**

In [2]:
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "agents").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")        # use the models already in the local cache; no network

FULL = os.environ.get("C4_FULL", "0") == "1"        # True: stage-1 recall on all 7 datasets (about 15 minutes)
REBUILD_PAIRS = False                               # True: regenerate data/reranker/*.jsonl and compare with the stored counts
RETRAIN_MINILM = False                              # True: fine-tune the MiniLM cross-encoder here (slow; may differ from Colab)

DATA = ROOT / "data"
RES = DATA / "results"
print("project root:", ROOT)
print("full run:", FULL, "| rebuild pairs:", REBUILD_PAIRS, "| retrain MiniLM:", RETRAIN_MINILM)

def show(title, header, rows, widths=None):
    print("\n" + title)
    cols = len(header)
    widths = widths or [max(len(str(x)) for x in [header[i]] + [r[i] for r in rows]) for i in range(cols)]
    print("  ".join(str(h).ljust(w) for h, w in zip(header, widths)))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print("  ".join(str(x).ljust(w) for x, w in zip(r, widths)))

project root: /Users/habbglobal/Desktop/J26-SE-327/project
full run: False | rebuild pairs: False | retrain MiniLM: False


## 1. Data: datasets and the train / test split

Ten datasets, all public. SMOS and eANCI (Italian) were translated to English (`data/translated_en`); eTour came already
translated. **Train** = iTrust, SMOS, eANCI, eTour. **Test** = LibEST, CM-1, EBT (never used in training). PetClinic is only the demo.

In [3]:
def count(name):
    base = DATA / "processed" / name
    arts = json.loads((base / "artifacts.json").read_text(encoding="utf-8"))
    reqs = sum(1 for a in arts if a["type"] == "requirement")
    links = len(json.loads((base / "traces.json").read_text(encoding="utf-8"))) if (base / "traces.json").exists() else None
    return reqs, len(arts) - reqs, links

groups = [("train", ["itrust", "smos", "eanci", "etour"]), ("test", ["libest", "cm1", "ebt"]), ("demo", ["petclinic"])]
rows = []
for g, names in groups:
    for n in names:
        r, c, l = count(n)
        rows.append([g, n, r, c, l if l is not None else "no gold (synthetic requirements)"])
show("Datasets (requirements, code/design artifacts, true links)", ["group", "dataset", "requirements", "targets", "true links"], rows)


Datasets (requirements, code/design artifacts, true links)
group  dataset    requirements  targets  true links                      
-----  ---------  ------------  -------  --------------------------------
train  itrust     131           392      399                             
train  smos       67            100      1044                            
train  eanci      39            55       567                             
train  etour      58            116      308                             
test   libest     52            14       204                             
test   cm1        22            53       45                              
test   ebt        41            50       98                              
demo   petclinic  26            37       no gold (synthetic requirements)


In [3]:
meta = json.loads((DATA / "reranker" / "meta.json").read_text(encoding="utf-8"))
print("Pair files written by agents/traceability_agent/reranker/data.py (seed", meta["seed"], ", stage-1 =", meta["retrieval"], ")")
print("rows:", meta["rows"], "| train label balance:", meta["train_label_balance"])
show("Per dataset", ["dataset", "requirements", "code", "true links", "train pairs (pos/hard/random)", "stage-1 recall@20"],
     [[k, v["requirements"], v["code"], v["true_links"], v.get("train_pairs", "-"), v.get("stage1_recall_at_k", "-")] for k, v in meta["datasets"].items()])

if REBUILD_PAIRS:
    from agents.traceability_agent.reranker.data import build_pairs
    new = build_pairs("hybrid")
    print("rebuilt rows:", new["rows"], "| identical to stored:", new["rows"] == meta["rows"])

Pair files written by agents/traceability_agent/reranker/data.py (seed 13 , stage-1 = hybrid )
rows: {'train': 4356, 'valid': 419, 'test_candidates': 1988} | train label balance: {'positive': 2124, 'negative': 2232}

Per dataset
dataset  requirements  code  true links  train pairs (pos/hard/random)             stage-1 recall@20
-------  ------------  ----  ----------  ----------------------------------------  -----------------
iTrust   131           392   399         {'pos': 361, 'hard': 600, 'random': 300}  -                
SMOS     67            100   1044        {'pos': 947, 'hard': 366, 'random': 183}  -                
eANCI    39            55    567         {'pos': 538, 'hard': 210, 'random': 105}  -                
eTour    58            116   308         {'pos': 278, 'hard': 312, 'random': 156}  -                
LibEST   52            14    204         -                                         1.0              
CM-1     22            53    45          -                      

## 2. Stage-1 baseline (the number every later method must beat)

Stage 1 = TF-IDF + `all-MiniLM-L6-v2`, fused by z-score (hybrid), best 60-word chunk per file. Candidates are the top-20 per requirement.
"top-k" means the k best of those; recall is against **all** true links of the dataset.

In [4]:
from agents.traceability_agent.reranker import evaluate as ev

cand = ev.load(DATA / "reranker" / "test_candidates.jsonl")
for r in cand:
    r["stage1"] = r["stage1_score"]
colab = json.loads((RES / "colab_results.json").read_text(encoding="utf-8"))
COLS = colab["columns"]

rows, worst = [], 0.0
for ds in ["CM-1", "EBT", "LibEST"]:
    sub = [r for r in cand if r["dataset"] == ds]
    m = ev.metrics(sub, meta["datasets"][ds]["true_links"], "stage1")
    mine = [m[c] for c in COLS]
    rows.append([ds] + ["%.3f" % x for x in mine])
    worst = max(worst, max(abs(a - b) for a, b in zip(mine, colab["stage1"][ds])))
show("Stage 1 only, held-out datasets", ["dataset"] + COLS, rows)
print("\nlargest difference to the Colab output:", round(worst, 4), "(should be < 0.002)")


Stage 1 only, held-out datasets
dataset  P@1    R@1    F1@1   P@3    R@3    F1@3   P@5    R@5    F1@5   P@10   R@10   F1@10  MRR  
-------  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----
CM-1     0.682  0.333  0.448  0.455  0.667  0.541  0.327  0.800  0.465  0.195  0.956  0.325  0.876
EBT      0.268  0.112  0.158  0.203  0.255  0.226  0.161  0.337  0.218  0.154  0.643  0.248  0.536
LibEST   0.558  0.142  0.227  0.526  0.402  0.456  0.454  0.578  0.509  0.377  0.961  0.541  0.767

largest difference to the Colab output: 0.0005 (should be < 0.002)


## 3. Is a code-specific *embedding* better at stage 1? (three models, recall of the shortlist)

Stage 1 uses one embedding model. Three were compared with the same TF-IDF part and the same data, no training:
* `all-MiniLM-L6-v2`: general English sentence model (the pipeline's choice),
* `st-codesearch-distilroberta-base`: trained for code search,
* `microsoft/unixcoder-base`: trained on code and its comments (plain mean pooling here; its special pooling mode was **not** tried).

The cell prints recall@10 of the embedding alone and of the hybrid (TF-IDF + embedding). The table recorded earlier is in
`data/results/stage1_embeddings.json`. Models that are not in the local cache are skipped with a message.

In [5]:
from agents.traceability_agent.config import CHUNK_WORDS
from agents.traceability_agent.preprocess import retrieval_text
from agents.traceability_agent.retrieval import embedding_matrix, hybrid_matrix, select_candidates, tfidf_matrix, zscore_fuse
from shared.embeddings.embedder import Embedder
from shared.schemas.traceability import Artifact

MODELS = {"MiniLM": "all-MiniLM-L6-v2",
          "code-search": "flax-sentence-embeddings/st-codesearch-distilroberta-base",
          "UniXcoder": "microsoft/unixcoder-base"}
NAMES = ["ebt", "libest", "cm1", "etour", "smos", "eanci", "itrust"] if FULL else ["ebt", "libest", "cm1", "etour"]
KS = (5, 10, 20)

def load_ds(name):
    base = DATA / "processed" / name
    arts = [Artifact(**a) for a in json.loads((base / "artifacts.json").read_text(encoding="utf-8"))]
    true = {(t["source_id"], t["target_id"]) for t in json.loads((base / "traces.json").read_text(encoding="utf-8"))}
    return [a for a in arts if a.type == "requirement"], [a for a in arts if a.type != "requirement"], true

def recall_at(s, t, sims, k, true):
    cands = select_candidates(s, t, sims, k, float("-inf"))
    return round(len({(c.source.id, c.target.id) for c in cands} & true) / len(true), 3)

tfidf_cache = {}
results = {}
for label, model in MODELS.items():
    try:
        emb = Embedder(model)
        emb.embed_batch(["test"])                      # fails here if the model is not cached
    except Exception as exc:
        print("skipped", label, "->", type(exc).__name__)
        continue
    for name in NAMES:
        s, t, true = load_ds(name)
        if name not in tfidf_cache:
            tfidf_cache[name] = tfidf_matrix(s, t, retrieval_text, CHUNK_WORDS)
        e_m = embedding_matrix(s, t, emb, retrieval_text, CHUNK_WORDS)
        h_m = zscore_fuse([tfidf_cache[name], e_m])          # = the pipeline's hybrid score
        results.setdefault(name, {})[label] = {
            "embedding": {k: recall_at(s, t, e_m, k, true) for k in KS},
            "hybrid": {k: recall_at(s, t, h_m, k, true) for k in KS},
            "tfidf": {k: recall_at(s, t, tfidf_cache[name], k, true) for k in KS}}
    print("done:", label, flush=True)

recorded = json.loads((RES / "stage1_embeddings.json").read_text(encoding="utf-8")) if (RES / "stage1_embeddings.json").exists() else {}
for kind in ("embedding", "hybrid"):
    rows = []
    for name in NAMES:
        row = [name]
        for label in MODELS:
            mine = results.get(name, {}).get(label, {}).get(kind, {}).get(10)
            rec = recorded.get(name, {}).get(label, {}).get(kind, {}).get("10")
            row.append("%s (rec. %s)" % ("%.3f" % mine if mine is not None else "-", rec if rec is not None else "-"))
        rows.append(row)
    show("Stage-1 recall@10, %s only   [this run (recorded)]" % kind, ["dataset"] + list(MODELS), rows)
if FULL:
    (RES / "stage1_embeddings.json").write_text(json.dumps(results, indent=1), encoding="utf-8")
    print("saved", RES / "stage1_embeddings.json")

done: MiniLM
done: code-search
done: UniXcoder

Stage-1 recall@10, embedding only   [this run (recorded)]
dataset  MiniLM              code-search         UniXcoder         
-------  ------------------  ------------------  ------------------
ebt      0.633 (rec. 0.633)  0.633 (rec. 0.633)  0.449 (rec. 0.449)
libest   0.971 (rec. 0.971)  0.961 (rec. 0.961)  0.941 (rec. 0.941)
cm1      0.933 (rec. 0.933)  0.889 (rec. 0.889)  0.689 (rec. 0.689)
etour    0.510 (rec. 0.51)   0.455 (rec. 0.455)  0.331 (rec. 0.331)
smos     0.259 (rec. 0.259)  0.249 (rec. 0.249)  0.266 (rec. 0.266)
eanci    0.236 (rec. 0.236)  0.220 (rec. 0.22)   0.213 (rec. 0.213)
itrust   0.333 (rec. 0.333)  0.328 (rec. 0.328)  0.331 (rec. 0.331)

Stage-1 recall@10, hybrid only   [this run (recorded)]
dataset  MiniLM              code-search         UniXcoder         
-------  ------------------  ------------------  ------------------
ebt      0.643 (rec. 0.643)  0.622 (rec. 0.622)  0.561 (rec. 0.561)
libest   0.961 (rec. 0

## 4. The cross-encoder as a second opinion (untrained MiniLM, computed here)

`cross-encoder/ms-marco-MiniLM-L-6-v2` reads the requirement and the code together and scores the pair. Its score is fused with
stage 1 per requirement: `fused = z(stage-1) + w * z(cross-encoder)`. A paired bootstrap over the requirements (2,000 resamples)
says whether the difference to stage 1 is distinguishable from noise: **an interval above 0 means supported**.
`hits@5` = true links found in the top 5 per requirement (recall@5 up to a constant).

In [6]:
from agents.traceability_agent.reranker import train as T
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CE = "cross-encoder/ms-marco-MiniLM-L-6-v2"
device = T.pick_device()
tok = AutoTokenizer.from_pretrained(CE)
ce = AutoModelForSequenceClassification.from_pretrained(CE).to(device).eval()
t0 = time.time()
scores = T.score_rows(ce, tok, cand, device, 512)
for r, s in zip(cand, scores):
    r["score"] = float(s)
print("scored %d (requirement, code) pairs on %s in %.0f s" % (len(cand), device, time.time() - t0))

W = (0.5, 1.0)
ev.add_fusion(cand, W)
rows, diffs = [], []
for ds in ["CM-1", "EBT", "LibEST"]:
    sub = [r for r in cand if r["dataset"] == ds]
    tot = meta["datasets"][ds]["true_links"]
    for key, label in [("stage1", "stage 1"), ("score", "cross-encoder alone"), ("fuse_0.5", "fused w=0.5"), ("fuse_1.0", "fused w=1.0")]:
        m = ev.metrics(sub, tot, key)
        rows.append([ds, label, "%.3f" % m["F1@3"], "%.3f" % m["R@5"], "%.3f" % m["R@10"], "%.3f" % m["MRR"]])
    for w, name in ((0.5, "fuse_0.5"), (1.0, "fuse_1.0")):
        mine = ev.metrics(sub, tot, name)
        rec = colab["minilm_untrained"]["fuse_%s" % w][ds]
        diffs.append(max(abs(mine[c] - rec[i]) for i, c in enumerate(COLS)))
show("Untrained MiniLM cross-encoder, held-out datasets", ["dataset", "method", "F1@3", "R@5", "R@10", "MRR"], rows)
print("\nlargest difference to the numbers recorded on Colab (CUDA vs this machine): %.3f" % max(diffs))

print("\nPaired bootstrap, fused minus stage 1: mean [95% interval]   (* = interval excludes 0)")
for w in W:
    boot = ev.bootstrap(cand, "stage1", "fuse_%s" % w)
    for name in ("ALL", "CM-1", "EBT", "LibEST"):
        cells = "   ".join("%s %+.3f [%+.3f, %+.3f]%s" % (m, v[0], v[1], v[2], "*" if v[1] > 0 or v[2] < 0 else " ") for m, v in boot[name].items())
        print("w=%s %-7s %s" % (w, name, cells))

scored 1988 (requirement, code) pairs on mps in 28 s

Untrained MiniLM cross-encoder, held-out datasets
dataset  method               F1@3   R@5    R@10   MRR  
-------  -------------------  -----  -----  -----  -----
CM-1     stage 1              0.541  0.800  0.956  0.876
CM-1     cross-encoder alone  0.414  0.756  0.867  0.736
CM-1     fused w=0.5          0.523  0.911  0.956  0.904
CM-1     fused w=1.0          0.541  0.867  0.956  0.851
EBT      stage 1              0.226  0.337  0.643  0.536
EBT      cross-encoder alone  0.244  0.398  0.561  0.517
EBT      fused w=0.5          0.226  0.388  0.663  0.574
EBT      fused w=1.0          0.281  0.398  0.673  0.579
LibEST   stage 1              0.456  0.578  0.961  0.767
LibEST   cross-encoder alone  0.389  0.598  0.907  0.702
LibEST   fused w=0.5          0.467  0.593  0.966  0.767
LibEST   fused w=1.0          0.478  0.583  0.961  0.776

largest difference to the numbers recorded on Colab (CUDA vs this machine): 0.000

Paired bootstr

## 5. Recorded Colab results (fine-tuned CodeBERT and fine-tuned MiniLM)

These were trained on a T4 GPU (CodeBERT 3 epochs, about 24 minutes; MiniLM about 5 minutes) on iTrust + SMOS + eANCI + eTour and tested on the three
held-out datasets. The numbers below are the recorded Colab output (`data/results/colab_results.json`).

In [7]:
tr = colab["training"]
show("Training log", ["model", "epoch", "train loss", "valid AUC", "seconds"],
     [[m, e["epoch"], e["train_loss"], e["valid_auc"], e["seconds"]] for m in tr for e in tr[m]])
print("Note: valid AUC uses validation requirements from the SAME projects as training, so it does not measure transfer to new projects.")

def row_for(model, method, ds):
    v = colab[model][method][ds] if method != "stage1" else colab["stage1"][ds]
    d = dict(zip(COLS, v))
    return ["%.3f" % d["F1@3"], "%.3f" % d["R@5"], "%.3f" % d["R@10"], "%.3f" % d["MRR"]]

for ds in ["CM-1", "EBT", "LibEST"]:
    rows = [["stage 1 only"] + row_for(None, "stage1", ds)]
    for model, label in [("codebert_finetuned", "CodeBERT (fine-tuned)"), ("minilm_finetuned", "MiniLM (fine-tuned)"), ("minilm_untrained", "MiniLM (untrained)")]:
        for method, mlabel in [("alone", "alone"), ("fuse_0.5", "fused w=0.5"), ("fuse_1.0", "fused w=1.0")]:
            rows.append(["%s, %s" % (label, mlabel)] + row_for(model, method, ds))
    show("Recorded Colab results on " + ds, ["method", "F1@3", "R@5", "R@10", "MRR"], rows)

print("\nRecorded bootstrap (fused minus stage 1), pooled over the 115 held-out requirements:")
for model in ("minilm_finetuned", "minilm_untrained"):
    for w in ("fuse_0.5", "fuse_1.0"):
        b = colab["bootstrap_vs_stage1"][model][w]["ALL"]
        print("%-17s %-8s " % (model, w) + "   ".join("%s %+.3f [%+.3f, %+.3f]%s" % (m, v[0], v[1], v[2], "*" if v[1] > 0 or v[2] < 0 else " ") for m, v in b.items()))


Training log
model             epoch  train loss  valid AUC  seconds
----------------  -----  ----------  ---------  -------
codebert          1      0.6422      0.7344     447    
codebert          2      0.5503      0.8074     451    
codebert          3      0.4748      0.8419     451    
minilm_finetuned  1      0.812       0.7677     91     
minilm_finetuned  2      0.5303      0.8209     91     
minilm_finetuned  3      0.4657      0.8233     91     
Note: valid AUC uses validation requirements from the SAME projects as training, so it does not measure transfer to new projects.

Recorded Colab results on CM-1
method                              F1@3   R@5    R@10   MRR  
----------------------------------  -----  -----  -----  -----
stage 1 only                        0.541  0.800  0.956  0.876
CodeBERT (fine-tuned), alone        0.054  0.156  0.444  0.211
CodeBERT (fine-tuned), fused w=0.5  0.523  0.756  0.889  0.762
CodeBERT (fine-tuned), fused w=1.0  0.414  0.711  0.822  0.59

In [8]:
# Optional: evaluate the real Colab score files if you copied them next to the project (data/reranker/colab_scores/<name>.jsonl),
# and optionally re-train the MiniLM cross-encoder here (slow; numbers may differ from Colab).
sc_dir = DATA / "reranker" / "colab_scores"
found = sorted(sc_dir.glob("*.jsonl")) if sc_dir.exists() else []
if found:
    for f in found:
        print("evaluating", f.name)
        rows = ev.load(f)
        for r in rows:
            r["stage1"] = r["stage1_score"]
        ev.add_fusion(rows, (0.5, 1.0))
        for ds in ["CM-1", "EBT", "LibEST"]:
            sub = [r for r in rows if r["dataset"] == ds]
            m = ev.metrics(sub, meta["datasets"][ds]["true_links"], "fuse_0.5")
            print("  ", ds, "fused w=0.5:", {k: round(m[k], 3) for k in ("F1@3", "R@5", "MRR")})
else:
    print("no Colab score files found (optional): the recorded Colab tables above are used instead")

if RETRAIN_MINILM:
    import argparse
    out = DATA / "reranker" / "local_model_minilm"
    T.train(argparse.Namespace(data=str(DATA / "reranker"), base=CE, out=str(out), epochs=3, batch=16, lr=2e-5,
                               max_length=512, max_steps=0, seed=13))
    T.predict(argparse.Namespace(data=str(DATA / "reranker"), model=str(out), out=None, max_length=512))
    rows = ev.load(out / "test_scores.jsonl")
    for r in rows:
        r["stage1"] = r["stage1_score"]
    ev.add_fusion(rows, (0.5,))
    for ds in ["CM-1", "EBT", "LibEST"]:
        m = ev.metrics([r for r in rows if r["dataset"] == ds], meta["datasets"][ds]["true_links"], "fuse_0.5")
        print("local fine-tuned MiniLM, fused w=0.5,", ds, {k: round(m[k], 3) for k in ("F1@3", "R@5", "MRR")})

no Colab score files found (optional): the recorded Colab tables above are used instead


## 6. PetClinic with the LLM: stage 1 versus stage 1 + reranker

26 synthetic requirements (written from the code) and 37 code files. Both methods shortlist `k` files per requirement; the reranker picks
them from a pool of 20. Every candidate pair was already judged by Llama 3.1 8B in earlier runs (`data/results/petclinic_verdicts.json`), so
no LLM call is needed here. *Confirmed* = the LLM said `is_linked` with confidence ≥ 0.5.

**Limits:** no human gold labels (the 8B LLM also makes false positives), one project, one run.

In [9]:
from agents.traceability_agent.agent import TraceabilityAgent
from agents.traceability_agent.reranker.rerank import CrossEncoderReranker

pc = json.loads((DATA / "processed" / "petclinic" / "artifacts.json").read_text(encoding="utf-8"))
arts = [Artifact(**a) for a in pc]
sources = [a for a in arts if a.type == "requirement"]
targets = [a for a in arts if a.type != "requirement"]
verd = json.loads((RES / "petclinic_verdicts.json").read_text(encoding="utf-8"))["pairs"]

def confirmed(pair):
    v = verd["%s||%s" % pair]
    return v["is_linked"] and v["confidence"] >= 0.5

embedder = Embedder("all-MiniLM-L6-v2")
rr = CrossEncoderReranker(CE, weight=0.5)
base = TraceabilityAgent(uskg=object(), embedder=embedder, llm=object(), retrieval="hybrid", project="spring-petclinic")
withrr = TraceabilityAgent(uskg=object(), embedder=embedder, llm=object(), retrieval="hybrid", project="spring-petclinic", reranker=rr, rerank_pool=20)

missing, rows = 0, []
sets = {}
for k in (3, 5):
    for name, agent in (("stage 1 only", base), ("stage 1 + reranker", withrr)):
        pairs = [(c.source.id, c.target.id) for c in agent.find_candidates(sources, targets, top_k=k)]
        sets[(k, name)] = set(pairs)
        known = [p for p in pairs if "%s||%s" % p in verd]
        missing += len(pairs) - len(known)
        yes = [p for p in known if confirmed(p)]
        rows.append(["top-%d" % k, name, len(pairs), len(yes), "%d/26" % len({p[0] for p in yes})])
show("PetClinic: what the LLM confirms among the shortlisted pairs (same number of LLM calls for both)",
     ["shortlist", "method", "LLM calls", "confirmed links", "requirements with >=1 link"], rows)
print("pairs without a stored verdict:", missing, "(should be 0)")

a, b = sets[(5, "stage 1 only")], sets[(5, "stage 1 + reranker")]
print("\ntop-5: %d pairs dropped by the reranker (LLM had confirmed %d of them), %d pairs added (LLM confirmed %d)" % (
    len(a - b), sum(confirmed(p) for p in a - b), len(b - a), sum(confirmed(p) for p in b - a)))
for p in sorted(b - a):
    if confirmed(p):
        print("   added and confirmed:", p[0], p[1].split("/")[-1])


PetClinic: what the LLM confirms among the shortlisted pairs (same number of LLM calls for both)
shortlist  method              LLM calls  confirmed links  requirements with >=1 link
---------  ------------------  ---------  ---------------  --------------------------
top-3      stage 1 only        78         38               23/26                     
top-3      stage 1 + reranker  78         39               23/26                     
top-5      stage 1 only        130        47               23/26                     
top-5      stage 1 + reranker  130        51               23/26                     
pairs without a stored verdict: 0 (should be 0)

top-5: 12 pairs dropped by the reranker (LLM had confirmed 0 of them), 12 pairs added (LLM confirmed 4)
   added and confirmed: PC-REQ-003 findOwners.html
   added and confirmed: PC-REQ-012 createOrUpdateVisitForm.html
   added and confirmed: PC-REQ-013 ownerDetails.html
   added and confirmed: PC-REQ-016 welcome.html


## 7. Query expansion (does an LLM that writes likely class names help stage 1?)

For each requirement the LLM (Llama 3.1 8B) was asked **once** for the class, method and file names that would probably implement it; these words were appended to the
requirement before retrieval. The stored answers are replayed here (`data/results/query_expansion_itrust.json`), so the
result is exactly reproducible without calling the LLM. Recall is stage-1 recall on the same requirements, with and without the extra words.

In [10]:
from agents.traceability_agent.reranker import query_expansion as qe

path = RES / "query_expansion_itrust.json"
if path.exists():
    out = qe.recall_table("itrust", embedder)
    stored = json.loads(path.read_text(encoding="utf-8"))
    print("LLM:", out["model"], "| requirements:", out["requirements"], "| true links:", out["true_links"])
    show("Stage-1 recall (hybrid) on iTrust", ["shortlist size", "plain requirement", "requirement + LLM words"],
         [["top-%s" % k, out["recall"]["plain"][k], out["recall"]["expanded"][k]] for k in out["recall"]["plain"]])
    errors = sum(1 for v in stored["answers"].values() if v.get("error"))
    print("answers with errors:", errors, "| seconds per LLM answer (mean): %.0f" % (sum(v["seconds"] for v in stored["answers"].values()) / len(stored["answers"])))
else:
    print("no stored answers (data/results/query_expansion_itrust.json): the experiment was not completed")

LLM: llama3.1-8k | requirements: 20 | true links: 68

Stage-1 recall (hybrid) on iTrust
shortlist size  plain requirement  requirement + LLM words
--------------  -----------------  -----------------------
top-5           0.338              0.338                  
top-10          0.441              0.441                  
top-20          0.544              0.529                  
answers with errors: 0 | seconds per LLM answer (mean): 29


## 8. My component's evaluation results (C4 Traceability Intelligence)

What is already measured in this repository (`data/results/summary.json`, `decay_injection_*.json`): retrieval recall of stage 1 on the
benchmarks, precision / recall / F1 of the full pipeline on a small iTrust sample, the decay-injection test, and the non-functional requirements.

In [11]:
S = json.loads((RES / "summary.json").read_text(encoding="utf-8"))
show("Benchmarks", ["dataset", "requirements", "targets", "true links", "language", "base rate"],
     [[k, v["requirements"], v["targets"], v["true_links"], v["language"], v["base_rate"]] for k, v in S["datasets"].items()])

rows = []
for ds, d in S["stage1_recall"].items():
    for k, v in d.items():
        rows.append([ds, "top-" + k, v["tfidf"], v["embeddings"], v["hybrid"]])
show("Stage-1 recall (no LLM)", ["dataset", "shortlist", "TF-IDF", "embeddings", "hybrid"], rows)

show("Precision / recall / F1 (these are small samples; see the dataset column)", ["dataset", "method", "precision", "recall", "F1"],
     [[x["dataset"], x["method"], x["precision"], x["recall"], x["f1"]] for x in S["prf"]])
print("note:", S["stage1_recall_notes"])

show("Non-functional requirements", ["requirement", "target", "measured", "met"],
     [[x["name"], x["target"], x["measured"], x["met"]] for x in S["nfr"]], widths=[45, 10, 60, 5])
print("\nstill pending according to summary.json:", S["pending"])

dec = sorted((RES).glob("decay_injection_*.json"))
if dec:
    D = json.loads(dec[-1].read_text(encoding="utf-8"))
    show("Decay injection (%s, %d known-good links, LLM %s)" % (dec[-1].name, D["sample_links"], D.get("model")),
         ["mutation", "injected", "detected"], [[k, v["injected"], v["detected"]] for k, v in D["per_mutation"].items()])
    print("decay detection accuracy: %.3f (target >= 0.85) | false decay rate on the control: %.3f" % (D["decay_detection_accuracy"], D["false_decay_rate"]))


Benchmarks
dataset  requirements  targets  true links  language       base rate
-------  ------------  -------  ----------  -------------  ---------
iTrust   131           392      399         Java/JSP       0.008    
eTour    58            116      308         Java           0.046    
LibEST   52            14       204         C              0.28     
CM-1     22            53       45          text (design)  0.039    

Stage-1 recall (no LLM)
dataset  shortlist  TF-IDF  embeddings  hybrid
-------  ---------  ------  ----------  ------
iTrust   top-10     0.373   0.331       0.416 
iTrust   top-20     0.521   0.436       0.556 
iTrust   top-50     0.659   0.617       0.684 
eTour    top-5      0.312   0.344       0.36  
eTour    top-10     0.468   0.51        0.532 
eTour    top-20     0.685   0.672       0.692 
eTour    top-50     0.873   0.877       0.909 
LibEST   top-3      0.407   0.461       0.422 
LibEST   top-5      0.578   0.613       0.588 
LibEST   top-10     0.951   0.96

In [12]:
# Optional: the live graph (needs Neo4j running locally). Skipped quietly if it is not available.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
    from apps.api import sources as live
    st = live.neo4j_status()
    print("Neo4j connected:", st["connected"])
    if st["connected"]:
        print(st["counts"])
        for p in live.neo4j_projects():
            print(" ", {k: p[k] for k in ("project", "requirements", "code_entities", "links", "decayed")})
except Exception as exc:
    print("graph section skipped:", type(exc).__name__)

Neo4j connected: True
{'requirements': 260, 'code_entities': 806, 'verified_traces': 260, 'decayed': 4, 'decay_flags': 4}
  {'project': 'CM-1', 'requirements': 24, 'code_entities': 53, 'links': 48, 'decayed': 0}
  {'project': 'eTour', 'requirements': 62, 'code_entities': 116, 'links': 4, 'decayed': 1}
  {'project': 'iTrust', 'requirements': 134, 'code_entities': 392, 'links': 130, 'decayed': 1}
  {'project': 'jbehave', 'requirements': 14, 'code_entities': 208, 'links': 14, 'decayed': 2}
  {'project': 'spring-petclinic', 'requirements': 26, 'code_entities': 37, 'links': 64, 'decayed': 0}


## 9. Conclusions (what the evidence supports, and what it does not)

1. **Stage 1 is the bottleneck.** TF-IDF + MiniLM finds only part of the true links in its shortlist (iTrust top-10: 41.6%, eTour 53%). The LLM can never recover a link
   that stage 1 did not shortlist.
2. **A fine-tuned CodeBERT cross-encoder did not generalise** to unseen projects: alone it is far worse than stage 1 (CM-1 MRR 0.211 versus 0.876); fused results are inconsistent. It is not used.
3. **A MiniLM cross-encoder fused with stage 1** puts about 0.1 to 0.16 more true links into the top 5 per requirement (pooled over 115 held-out requirements, 95% interval above 0), about 10%.
   MRR and top-3 quality are **not** reliably improved.
4. **Fine-tuning added nothing demonstrable:** the untrained model gives a similar gain. The honest reading is "an off-the-shelf cross-encoder as a second opinion helps a little".
5. **Code-specific embeddings (code-search model, UniXcoder) did not beat MiniLM** at stage 1 on 7 datasets (UniXcoder tested with plain mean pooling only).
6. **PetClinic with Llama 3.1 8B:** at the same number of LLM calls (top-5, 130) the reranker leads to 51 instead of 47 LLM-confirmed links; every dropped pair had been rejected by the LLM. No human gold, one project.
7. **Query expansion with the LLM did not help** (20 iTrust requirements: top-10 recall 0.441 before and after); the LLM guesses class names that are not in the code.
8. **Open problems in my component:** LLM verification precision (iTrust 37.5%) and latency (about 52 s per call on the CPU server) are far from the targets, and the decay-injection detection (69.6% on 9 links) is below its 85% target.

**Limits to say out loud:** weights chosen after seeing the test results; small test sets (22, 41, 52 requirements); eTour code is Italian; SMOS has hub classes linked to almost every requirement;
LLM verdicts are noisy; the fine-tuned results come from Colab and are only replayed here.